# Laboratorio: Isolation Forest sobre **4 KPIs** unidos (AIOps)

En este notebook unimos **cuatro métricas** del AIOps Challenge 2018 que comparten la misma ventana temporal (~146k puntos, 1 minuto, abr–jul 2017). Cada fila es un instante; cada columna es el `value` de un KPI distinto.

El objetivo es el mismo que en el laboratorio de un solo KPI: entrenar **Isolation Forest** multivariado, interpretar el ranking y comparar hiperparámetros — pero ahora las anomalías son **combinaciones raras entre métricas**, no solo rareza en una serie.

Dataset: [NetManAIOps / KPI-Anomaly-Detection](https://github.com/NetManAIOps/KPI-Anomaly-Detection) (competencia AIOps 2018).


## 0. Dependencias

Solo hace falta lo del bloque de imports. Los datos se descargan por HTTP si no están en `data/`.


In [ ]:
# Sin instalaciones extra.


## 1. Imports y configuración

Usamos `pandas` para el join por `timestamp`, `scikit-learn` para Isolation Forest y `StandardScaler` (imprescindible al mezclar KPIs con escalas distintas).


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.tree import plot_tree

RANDOM_STATE = 42
plt.style.use("default")
pd.set_option("display.max_columns", 40)


## 2. Descargar dataset AIOps KPI

Mismo `phase2_train.csv` que en el laboratorio de un KPI. Tiene **29 KPI IDs**; solo usaremos **cuatro** del grupo “largo” (misma época, timestamps alineables).


In [ ]:
import urllib.request
import zipfile

KPI_ZIP_URL = (
    "https://github.com/NetManAIOps/KPI-Anomaly-Detection/raw/master/"
    "Finals_dataset/phase2_train.csv.zip"
)
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
kpi_zip_path = DATA_DIR / "phase2_train.csv.zip"
kpi_csv_path = DATA_DIR / "phase2_train.csv"

if not kpi_csv_path.exists():
    print("Descargando AIOps KPI (puede tardar ~30 s)...")
    with urllib.request.urlopen(KPI_ZIP_URL, timeout=120) as resp:
        kpi_zip_path.write_bytes(resp.read())
    with zipfile.ZipFile(kpi_zip_path) as zf:
        zf.extract("phase2_train.csv", path=DATA_DIR)
    print("Guardado en:", kpi_csv_path.resolve())
else:
    print("Usando cache local:", kpi_csv_path.resolve())

kpi_all = pd.read_csv(kpi_csv_path)
kpi_all["timestamp"] = pd.to_datetime(kpi_all["timestamp"], unit="s")

kpi_summary = (
    kpi_all.groupby("KPI ID")
    .agg(
        puntos=("value", "size"),
        desde=("timestamp", "min"),
        hasta=("timestamp", "max"),
    )
    .sort_values("puntos", ascending=False)
)
print(f"KPIs disponibles: {len(kpi_summary)}  |  Filas totales: {len(kpi_all):,}")
kpi_summary.head(10)


## 3. Join de 4 KPIs por `timestamp`

Elegimos cuatro IDs del grupo con ~146k puntos (abr–jul 2017). El join es un `pivot`: una fila por minuto, una columna por KPI.

| Alias | KPI ID (primeros 8 chars) |
|-------|---------------------------|
| kpi_a | `05f10d3a...` |
| kpi_b | `c69a50cf...` |
| kpi_c | `1c6d7a26...` |
| kpi_d | `adb2fde9...` |

Usamos `dropna()` (join interno): descartamos minutos donde falte algún KPI. En la práctica pierde muy pocas filas.


In [ ]:
KPI_IDS = ['05f10d3a-239c-3bef-9bdc-a2feeb0037aa', 'c69a50cf-ee03-3bd7-831e-407d36c7ee91', '1c6d7a26-1f1a-3321-bb4d-7a9d969ec8f0', 'adb2fde9-8589-3f5b-a410-5fe14386c7af']
SHORT_NAMES = ['kpi_a', 'kpi_b', 'kpi_c', 'kpi_d']
ID_TO_SHORT = dict(zip(KPI_IDS, SHORT_NAMES))

subset = kpi_all[kpi_all["KPI ID"].isin(KPI_IDS)].copy()
wide = (
    subset.pivot(index="timestamp", columns="KPI ID", values="value")
    .sort_index()
    .rename(columns=ID_TO_SHORT)
    .reset_index()
)
raw = wide.dropna().reset_index(drop=True)

print("Filas despues del join:", len(raw))
print("Columnas KPI:", SHORT_NAMES)
print(f"Desde {raw['timestamp'].min()} hasta {raw['timestamp'].max()}")
raw[SHORT_NAMES].describe().T


## 4. Primera mirada

Cuatro series en paralelo. Las escalas suelen diferir (el CSV no trae nombres legibles de métrica); por eso luego **estandarizamos** antes del IF.


In [ ]:
fig, axes = plt.subplots(len(SHORT_NAMES), 1, figsize=(13, 8), sharex=True)

for ax, name in zip(axes, SHORT_NAMES):
    ax.plot(raw["timestamp"], raw[name], color="0.25", linewidth=0.9)
    ax.set_ylabel(name)
    ax.grid(True, alpha=0.3)

axes[0].set_title("Cuatro KPIs unidos por timestamp (valores en escala original)")
axes[-1].set_xlabel("timestamp")
plt.tight_layout()
plt.show()


## 5. Construcción de features

Por cada KPI calculamos `value`, `diff_1` y `rolling_z` (ventana 12). Agregamos `hour` y `dayofweek` compartidos.

En total: 4 × 3 + 2 = **14 features** para el Isolation Forest multivariado.


### Qué es `rolling_z` (z-score móvil)

Para cada KPI, compara el valor actual con la media y el desvío de los últimos 12 minutos. Un pico en **una** métrica con las demás normales puede ser anómalo a nivel sistema.


In [ ]:
def build_features_multikpi(df: pd.DataFrame, kpi_cols: list[str], window: int = 12) -> pd.DataFrame:
    data = df.copy()
    data["timestamp"] = pd.to_datetime(data["timestamp"])
    data = data.sort_values("timestamp").reset_index(drop=True)

    features = pd.DataFrame({"timestamp": data["timestamp"]})
    for col in kpi_cols:
        y = data[col].astype(float)
        features[col] = y
        features[f"{col}_diff_1"] = y.diff()
        roll_mean = y.rolling(window, min_periods=3).mean()
        roll_std = y.rolling(window, min_periods=3).std()
        features[f"{col}_rolling_z"] = (y - roll_mean) / roll_std.replace(0, np.nan)

    features["hour"] = features["timestamp"].dt.hour
    features["dayofweek"] = features["timestamp"].dt.dayofweek
    return features.dropna().reset_index(drop=True)


features = build_features_multikpi(raw, SHORT_NAMES, window=12)
features.head()


## 6. Matriz de entrenamiento

Mezclamos KPIs con unidades distintas → `StandardScaler` sobre todas las columnas numéricas.


In [ ]:
feature_cols = ['kpi_a', 'kpi_a_diff_1', 'kpi_a_rolling_z', 'kpi_b', 'kpi_b_diff_1', 'kpi_b_rolling_z', 'kpi_c', 'kpi_c_diff_1', 'kpi_c_rolling_z', 'kpi_d', 'kpi_d_diff_1', 'kpi_d_rolling_z', 'hour', 'dayofweek']

X = features[feature_cols]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Filas para entrenar:", X.shape[0])
print("N features:", len(feature_cols))


## 7. Entrenar Isolation Forest

Parámetros principales (se fijan **al crear y entrenar** el modelo con `fit()`):

- `n_estimators = m`: cantidad de árboles.
- `max_samples = ψ`: tamaño de submuestra por árbol.
- `contamination`: proporción del ranking que se etiqueta como anomalía.

Entrenamos **tres configuraciones** sobre la misma matriz para comparar cómo cambian las etiquetas.


In [ ]:
psi = min(256, len(features))

MODEL_CONFIGS = [
    {"name": "base", "label": "base (m=200, contamination=2%)", "m": 200, "contamination": 0.02},
    {"name": "pocas_alertas", "label": "pocas alertas (m=200, contamination=0.5%)", "m": 200, "contamination": 0.005},
    {"name": "pocos_arboles", "label": "pocos arboles (m=50, contamination=2%)", "m": 50, "contamination": 0.02},
]


def fit_isolation_forest(X_scaled, m, psi, contamination):
    clf = IsolationForest(
        n_estimators=m,
        max_samples=psi,
        contamination=contamination,
        random_state=RANDOM_STATE,
    )
    pred = clf.fit_predict(X_scaled)
    scores = -clf.score_samples(X_scaled)
    return clf, scores, pred == -1


models = {}
for cfg in MODEL_CONFIGS:
    clf, scores, flags = fit_isolation_forest(
        X_scaled, cfg["m"], psi, cfg["contamination"]
    )
    models[cfg["name"]] = {
        "model": clf,
        "label": cfg["label"],
        "m": cfg["m"],
        "contamination": cfg["contamination"],
        "anomaly_score": scores,
        "is_anomaly": flags,
    }

base = models["base"]
model = base["model"]
m = base["m"]
contamination = base["contamination"]

results = features.copy()
results["anomaly_score"] = base["anomaly_score"]
results["is_anomaly"] = base["is_anomaly"]
for cfg in MODEL_CONFIGS:
    if cfg["name"] == "base":
        continue
    results[f"is_anomaly_{cfg['name']}"] = models[cfg["name"]]["is_anomaly"]

# Promedio de valores (solo para graficos; el IF usa todas las features)
results["value_mean"] = results[SHORT_NAMES].mean(axis=1)

comparison_rows = []
for cfg in MODEL_CONFIGS:
    if cfg["name"] == "base":
        continue
    flags = models[cfg["name"]]["is_anomaly"]
    comparison_rows.append(
        {
            "config": cfg["name"],
            "m": cfg["m"],
            "contamination": cfg["contamination"],
            "n_anomalies": int(flags.sum()),
            "overlap_con_base": int((flags & base["is_anomaly"]).sum()),
            "pct_etiquetas_distintas": round(100 * (flags != base["is_anomaly"]).mean(), 2),
        }
    )

display(pd.DataFrame(comparison_rows))
results[["timestamp", "value_mean", "anomaly_score", "is_anomaly"]].head()


### Comparar las tres configuraciones

Misma matriz multivariada: `contamination` cambia **cuántos** puntos se etiquetan; `m` distinto con `contamination` fijo cambia **cuáles**.


In [ ]:
fig, axes = plt.subplots(len(MODEL_CONFIGS), 1, figsize=(14, 9), sharex=True)

for ax, cfg in zip(axes, MODEL_CONFIGS):
    flags = models[cfg["name"]]["is_anomaly"]
    n_flags = int(flags.sum())
    ax.plot(results["timestamp"], results["value_mean"], color="0.25", linewidth=0.9, label="promedio 4 KPIs")
    ax.scatter(
        results.loc[flags, "timestamp"],
        results.loc[flags, "value_mean"],
        color="tab:red",
        s=22,
        zorder=3,
        label=f"anomalias ({n_flags})",
    )
    ax.set_title(cfg["label"])
    ax.set_ylabel("value_mean")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("timestamp")
fig.suptitle("Tres modelos IF sobre 4 KPIs unidos", y=1.01)
plt.tight_layout()
plt.show()


## 8. Resultados: ranking y visualización

Top anomalías con los valores de los cuatro KPIs en ese minuto.


In [ ]:
display_cols = ["timestamp", "anomaly_score", "is_anomaly"] + ['kpi_a', 'kpi_b', 'kpi_c', 'kpi_d']
top_anomalies = results.sort_values("anomaly_score", ascending=False).head(15)
top_anomalies[display_cols]


In [ ]:
def annotate_top_points(ax, data, y_col, score_col="anomaly_score", n=3):
    top = data.sort_values(score_col, ascending=False).head(n)
    for rank, (_, row) in enumerate(top.iterrows(), start=1):
        ax.annotate(
            f"#{rank}",
            xy=(row["timestamp"], row[y_col]),
            xytext=(6, 8),
            textcoords="offset points",
            fontsize=9,
            color="tab:red",
            arrowprops={"arrowstyle": "->", "color": "tab:red", "lw": 0.8},
        )


threshold = results.loc[results["is_anomaly"], "anomaly_score"].min()
n_anomalies = int(results["is_anomaly"].sum())

fig, axes = plt.subplots(5, 1, figsize=(14, 14), sharex=True, gridspec_kw={"height_ratios": [1.5, 1.5, 1.5, 1.5, 1.2]})

for ax, name in zip(axes[:4], SHORT_NAMES):
    ax.plot(results["timestamp"], results[name], linewidth=0.9, color="0.25", label=name)
    ax.scatter(
        results.loc[results["is_anomaly"], "timestamp"],
        results.loc[results["is_anomaly"], name],
        color="tab:red",
        s=28,
        zorder=3,
        label=f"anomalia IF ({n_anomalies})",
    )
    if name == SHORT_NAMES[0]:
        annotate_top_points(ax, results, y_col=name, n=3)
    ax.set_ylabel(name)
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_title(
    "Isolation Forest multivariado (4 KPIs)\n"
    "Rojo = minuto anomalo segun la combinacion de las 14 features"
)

ax = axes[4]
ax.plot(results["timestamp"], results["anomaly_score"], linewidth=1, color="tab:orange", label="anomaly_score")
ax.axhline(threshold, color="tab:red", linestyle="--", linewidth=1.2, label="umbral por contamination")
ax.fill_between(
    results["timestamp"],
    threshold,
    results["anomaly_score"],
    where=results["anomaly_score"] >= threshold,
    color="tab:red",
    alpha=0.15,
    interpolate=True,
)
ax.set_ylabel("score")
ax.set_xlabel("timestamp")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 9. Visualizar un iTree

Proyectamos en 2D: `kpi_a` vs `kpi_b` (estandarizados). El árbol particiona el plano según **combinaciones** de ambas métricas.


In [ ]:
tree_cols = ["kpi_a", "kpi_b"]
X_tree_raw = results[tree_cols]
tree_scaler = StandardScaler()
X_tree = tree_scaler.fit_transform(X_tree_raw)

one_tree_if = IsolationForest(
    n_estimators=1,
    max_samples=min(64, len(results)),
    contamination=contamination,
    random_state=7,
)
one_tree_if.fit(X_tree)
one_tree_pred = one_tree_if.predict(X_tree) == -1
single_tree = one_tree_if.estimators_[0]

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

plot_tree(
    single_tree,
    max_depth=3,
    feature_names=["kpi_a (std)", "kpi_b (std)"],
    filled=True,
    rounded=True,
    impurity=False,
    fontsize=8,
    ax=axes[0],
)
axes[0].set_title("Un iTree: cortes sobre kpi_a y kpi_b (estandarizados)")

x_min, x_max = X_tree[:, 0].min() - 0.5, X_tree[:, 0].max() + 0.5
y_min, y_max = X_tree[:, 1].min() - 0.5, X_tree[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
grid = np.c_[xx.ravel(), yy.ravel()]
grid_score = -one_tree_if.score_samples(grid).reshape(xx.shape)

contour = axes[1].contourf(xx, yy, grid_score, levels=12, cmap="YlOrRd", alpha=0.75)
plt.colorbar(contour, ax=axes[1], label="score (mayor = mas aislado)")
axes[1].scatter(X_tree[~one_tree_pred, 0], X_tree[~one_tree_pred, 1], s=14, color="0.25", alpha=0.65, label="normal")
axes[1].scatter(X_tree[one_tree_pred, 0], X_tree[one_tree_pred, 1], s=28, color="tab:red", label="anomalo")
axes[1].set_xlabel("kpi_a estandarizado")
axes[1].set_ylabel("kpi_b estandarizado")
axes[1].set_title("Regiones del iTree en el plano kpi_a vs kpi_b")
axes[1].legend(loc="upper left")
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()


## 10. Comparación con umbral simple

Umbral univariado: percentiles 1% y 99% sobre el **promedio** de los cuatro KPIs (`value_mean`). El IF usa las 14 features; suelen detectar cosas distintas.


In [ ]:
q_low, q_high = results["value_mean"].quantile([0.01, 0.99])
results["univariate_outlier"] = (results["value_mean"] < q_low) | (results["value_mean"] > q_high)

pd.crosstab(
    results["univariate_outlier"],
    results["is_anomaly"],
    rownames=["umbral_univariado"],
    colnames=["isolation_forest"],
)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})

axes[0].plot(results["timestamp"], results["value_mean"], linewidth=1, color="0.35", label="promedio 4 KPIs")
axes[0].axhline(q_low, color="tab:blue", linestyle=":", linewidth=1.2)
axes[0].axhline(q_high, color="tab:blue", linestyle=":", linewidth=1.2, label="percentil 1% y 99%")
axes[0].scatter(
    results.loc[results["univariate_outlier"], "timestamp"],
    results.loc[results["univariate_outlier"], "value_mean"],
    s=55,
    facecolors="none",
    edgecolors="tab:blue",
    linewidths=1.4,
    label="umbral univariado",
)
axes[0].scatter(
    results.loc[results["is_anomaly"], "timestamp"],
    results.loc[results["is_anomaly"], "value_mean"],
    s=24,
    color="tab:red",
    label="Isolation Forest",
    zorder=3,
)
axes[0].set_title("Umbral sobre promedio vs IF multivariado (4 KPIs)")
axes[0].set_ylabel("value_mean")
axes[0].legend(loc="upper left")
axes[0].grid(True, alpha=0.3)

category = np.select(
    [
        results["univariate_outlier"] & results["is_anomaly"],
        results["is_anomaly"],
        results["univariate_outlier"],
    ],
    ["ambos", "solo IF", "solo umbral"],
    default="ninguno",
)
category_to_y = {"ninguno": 0, "solo umbral": 1, "solo IF": 2, "ambos": 3}
category_colors = {"ninguno": "0.85", "solo umbral": "tab:blue", "solo IF": "tab:red", "ambos": "purple"}

for cat, y in category_to_y.items():
    mask = category == cat
    axes[1].scatter(
        results.loc[mask, "timestamp"],
        np.full(mask.sum(), y),
        s=12 if cat == "ninguno" else 28,
        color=category_colors[cat],
        label=cat,
        alpha=0.8,
    )
axes[1].set_yticks(list(category_to_y.values()), list(category_to_y.keys()))
axes[1].set_xlabel("timestamp")
axes[1].legend(loc="upper left", ncol=4)
axes[1].grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plt.show()


## 11. Sensibilidad a `contamination`

Misma matriz `X_scaled`; solo cambia el corte del ranking.


In [ ]:
contamination_values = [0.005, 0.01, 0.02, 0.05]
summary_rows = []

fig, axes = plt.subplots(len(contamination_values), 1, figsize=(14, 10), sharex=True, sharey=True)
fig.suptitle("Sensibilidad a contamination (4 KPIs unidos)", y=1.02)

for ax, cont in zip(axes, contamination_values):
    tmp_model = IsolationForest(
        n_estimators=m,
        max_samples=psi,
        contamination=cont,
        random_state=RANDOM_STATE,
    )
    tmp_pred = tmp_model.fit_predict(X_scaled) == -1
    n_tmp = int(tmp_pred.sum())
    summary_rows.append({"contamination": cont, "n_anomalies": n_tmp})

    ax.plot(results["timestamp"], results["value_mean"], linewidth=1, color="0.35")
    ax.scatter(
        results.loc[tmp_pred, "timestamp"],
        results.loc[tmp_pred, "value_mean"],
        color="tab:red",
        s=24,
        zorder=3,
    )
    ax.set_ylabel(f"{cont:.1%}\n({n_tmp} pts)")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("timestamp")
plt.tight_layout()
plt.show()

pd.DataFrame(summary_rows)


## 12. Sensibilidad a $\psi$

Con `contamination` fijo, el **número** de anomalías no cambia; miramos overlap del top-30 y % de etiquetas distintas.


In [ ]:
psi_values = [64, 128, 256, min(512, len(X_scaled))]
psi_rows = []
base_top_idx = set(results.sort_values("anomaly_score", ascending=False).head(30).index)
base_flags = results["is_anomaly"].to_numpy()
n_esperado = int(contamination * len(X_scaled))

for psi_candidate in psi_values:
    tmp_model = IsolationForest(
        n_estimators=m,
        max_samples=psi_candidate,
        contamination=contamination,
        random_state=RANDOM_STATE,
    )
    tmp_pred = tmp_model.fit_predict(X_scaled) == -1
    tmp_score = -tmp_model.score_samples(X_scaled)
    tmp_top_idx = set(
        pd.Series(tmp_score, index=results.index)
        .sort_values(ascending=False)
        .head(30)
        .index
    )
    psi_rows.append({
        "psi": psi_candidate,
        "n_anomalies": int(tmp_pred.sum()),
        "n_esperado_por_cont": n_esperado,
        "overlap_top30_vs_base": len(base_top_idx & tmp_top_idx) / len(base_top_idx),
        "pct_etiquetas_distintas": float((tmp_pred != base_flags).mean()),
    })

pd.DataFrame(psi_rows)


## 13. Sensibilidad a $m$

Más árboles suelen estabilizar el ranking; el costo de entrenamiento crece.


In [ ]:
m_values = [1, 50, 500, 1500, 8000]
m_rows = []
base_top_idx = set(results.sort_values("anomaly_score", ascending=False).head(30).index)
base_flags = results["is_anomaly"].to_numpy()
n_esperado = int(contamination * len(X_scaled))

for m_candidate in m_values:
    tmp_model = IsolationForest(
        n_estimators=m_candidate,
        max_samples=psi,
        contamination=contamination,
        random_state=RANDOM_STATE,
    )
    tmp_pred = tmp_model.fit_predict(X_scaled) == -1
    tmp_score = -tmp_model.score_samples(X_scaled)
    tmp_top_idx = set(
        pd.Series(tmp_score, index=results.index)
        .sort_values(ascending=False)
        .head(30)
        .index
    )
    m_rows.append({
        "m": m_candidate,
        "n_anomalies": int(tmp_pred.sum()),
        "n_esperado_por_cont": n_esperado,
        "overlap_top30_vs_base": len(base_top_idx & tmp_top_idx) / len(base_top_idx),
        "pct_etiquetas_distintas": float((tmp_pred != base_flags).mean()),
    })

pd.DataFrame(m_rows)


## 14. Interpretar anomalías con features

Revisamos el top-10 con los valores de cada KPI y sus `rolling_z`.


In [ ]:
display_cols = (
    ["timestamp", "anomaly_score", "is_anomaly"]
    + SHORT_NAMES
    + [f"{c}_rolling_z" for c in SHORT_NAMES]
)
results.sort_values("anomaly_score", ascending=False).head(10)[display_cols]


## 15. Ejercicios para obligatorio

1. Reemplazar un `KPI_ID` por otro del `kpi_summary` con ~146k puntos. ¿Cambia el overlap temporal tras el join?
2. Agregar un quinto KPI joinable. ¿Cuántas features tendría el modelo?
3. Comparar `contamination=0.01` vs `0.05` si cada alerta implica revisión manual de un SRE.
4. ¿Por qué es obligatorio escalar antes de mezclar KPIs con escalas distintas?


## Ejercicio 15.1: Reemplazar un KPI por otro con ~146k puntos

Reemplazamos `kpi_d` (`adb2fde9...`) por `42d6616d-c9c5-370a-a8ba-17ead74f3114` (146253 puntos).

Pregunta: ¿Cambia el overlap temporal tras el join?

In [ ]:
# Ejercicio 15.1 - Reemplazar kpi_d por otro KPI

NEW_KPI_ID = '42d6616d-c9c5-370a-a8ba-17ead74f3114'

KPI_IDS_NEW = [
    '05f10d3a-239c-3bef-9bdc-a2feeb0037aa',  # kpi_a (sin cambio)
    'c69a50cf-ee03-3bd7-831e-407d36c7ee91',   # kpi_b (sin cambio)
    '1c6d7a26-1f1a-3321-bb4d-7a9d969ec8f0',   # kpi_c (sin cambio)
    NEW_KPI_ID,                                 # kpi_d_new (reemplazo)
]
SHORT_NAMES_NEW = ['kpi_a', 'kpi_b', 'kpi_c', 'kpi_d_new']
ID_TO_SHORT_NEW = dict(zip(KPI_IDS_NEW, SHORT_NAMES_NEW))

subset_new = kpi_all[kpi_all["KPI ID"].isin(KPI_IDS_NEW)].copy()
wide_new = (
    subset_new.pivot(index="timestamp", columns="KPI ID", values="value")
    .sort_index()
    .rename(columns=ID_TO_SHORT_NEW)
    .reset_index()
)
raw_new = wide_new.dropna().reset_index(drop=True)

# Comparar overlap temporal
print("=== ORIGINAL ===")
print(f"Filas: {len(raw)}")
print(f"Desde: {raw['timestamp'].min()}  Hasta: {raw['timestamp'].max()}")

print("\n=== CON REEMPLAZO ===")
print(f"Filas: {len(raw_new)}")
print(f"Desde: {raw_new['timestamp'].min()}  Hasta: {raw_new['timestamp'].max()}")

print(f"\nDiferencia de filas: {len(raw) - len(raw_new)}")
print(f"Timestamps en comun: {len(set(raw['timestamp']) & set(raw_new['timestamp']))}")

## Ejercicio 15.2: Agregar un quinto KPI joinable

Agregamos `8723f0fb-eaef-32e6-b372-6034c9c04b80` (146243 pts) como `kpi_e`.

Con 5 KPIs: 5 × 3 + 2 = **17 features**.

In [ ]:
# Ejercicio 15.2 - Agregar kpi_e (quinto KPI)

FIFTH_KPI_ID = '8723f0fb-eaef-32e6-b372-6034c9c04b80'

KPI_IDS_5 = KPI_IDS + [FIFTH_KPI_ID]
SHORT_NAMES_5 = ['kpi_a', 'kpi_b', 'kpi_c', 'kpi_d', 'kpi_e']
ID_TO_SHORT_5 = dict(zip(KPI_IDS_5, SHORT_NAMES_5))

subset_5 = kpi_all[kpi_all["KPI ID"].isin(KPI_IDS_5)].copy()
wide_5 = (
    subset_5.pivot(index="timestamp", columns="KPI ID", values="value")
    .sort_index()
    .rename(columns=ID_TO_SHORT_5)
    .reset_index()
)
raw_5 = wide_5.dropna().reset_index(drop=True)

# Construir features con 5 KPIs
features_5 = build_features_multikpi(raw_5, SHORT_NAMES_5, window=12)
feature_cols_5 = [c for c in features_5.columns if c != "timestamp"]

print(f"Filas: {len(features_5)}")
print(f"N features: {len(feature_cols_5)}")
print(f"Features: {feature_cols_5}")

# Entrenar IF con 5 KPIs
X_5 = features_5[feature_cols_5]
scaler_5 = StandardScaler()
X_5_scaled = scaler_5.fit_transform(X_5)

clf_5 = IsolationForest(
    n_estimators=200,
    max_samples=min(256, len(features_5)),
    contamination=0.02,
    random_state=RANDOM_STATE,
)
pred_5 = clf_5.fit_predict(X_5_scaled)
n_anomalies_5 = int((pred_5 == -1).sum())

print(f"\nAnomalias detectadas (5 KPIs): {n_anomalies_5}")
print(f"Anomalias con 4 KPIs (base):   {int(results['is_anomaly'].sum())}")

## Ejercicio 15.3: Comparar contamination=0.01 vs 0.05

Contexto: cada alerta genera revisión manual de un SRE. Se compara la cantidad de alertas, el overlap entre ambos modelos, y el trade-off precisión vs cobertura.

In [ ]:
# Ejercicio 15.3 - contamination=0.01 vs 0.05

configs_3 = {"conservador (1%)": 0.01, "agresivo (5%)": 0.05}
results_3 = {}

for label, cont in configs_3.items():
    mdl = IsolationForest(
        n_estimators=m, max_samples=psi, contamination=cont, random_state=RANDOM_STATE
    )
    pred = mdl.fit_predict(X_scaled) == -1
    scores = -mdl.score_samples(X_scaled)
    results_3[label] = {"pred": pred, "scores": scores, "n": int(pred.sum()), "cont": cont}

# Tabla comparativa
print("=== Comparacion ===")
for label, r in results_3.items():
    print(f"{label}: {r['n']} alertas ({r['cont']:.0%} de {len(X_scaled)})")

overlap = int((results_3["conservador (1%)"]["pred"] & results_3["agresivo (5%)"]["pred"]).sum())
solo_agresivo = results_3["agresivo (5%)"]["n"] - overlap
print(f"\nOverlap: {overlap} alertas en comun")
print(f"Alertas exclusivas del 5%: {solo_agresivo}")
print(f"\nCon 1%: un SRE revisa ~{results_3['conservador (1%)']['n']} alertas en 3.5 meses")
print(f"Con 5%: un SRE revisa ~{results_3['agresivo (5%)']['n']} alertas en 3.5 meses")
print(f"Diferencia: {solo_agresivo} revisiones adicionales")

# Grafico comparativo
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

for ax, (label, r) in zip(axes, results_3.items()):
    ax.plot(results["timestamp"], results["value_mean"], linewidth=0.9, color="0.35")
    ax.scatter(
        results.loc[r["pred"], "timestamp"],
        results.loc[r["pred"], "value_mean"],
        color="tab:red", s=22, zorder=3, label=f"anomalias ({r['n']})"
    )
    ax.set_title(f"{label}")
    ax.set_ylabel("value_mean")
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("timestamp")
plt.tight_layout()
plt.show()

## Ejercicio 15.4: ¿Por qué es obligatorio escalar antes de mezclar KPIs?

Isolation Forest particiona el espacio con cortes aleatorios sobre cada feature. Si los KPIs tienen escalas distintas (ej: uno va de 0 a 1, otro de 0 a 10000), el algoritmo va a hacer la mayoría de cortes "útiles" sobre la feature con mayor rango numérico, ignorando las demás.

Sin escalar, un KPI con valores grandes **domina** las particiones y las anomalías en KPIs de escala chica se vuelven invisibles.

`StandardScaler` lleva todas las features a media=0 y desvío=1, dándoles el mismo peso en las particiones del bosque.

In [ ]:
# Ejercicio 15.4 - Demostración: IF sin escalar vs con escalar

# Sin escalar
clf_no_scale = IsolationForest(
    n_estimators=m, max_samples=psi, contamination=contamination, random_state=RANDOM_STATE
)
pred_no_scale = clf_no_scale.fit_predict(X) == -1  # X sin escalar

# Con escalar (ya tenemos results["is_anomaly"])
pred_scaled = results["is_anomaly"].to_numpy()

overlap = int((pred_no_scale & pred_scaled).sum())
pct_diff = round(100 * (pred_no_scale != pred_scaled).mean(), 2)

print(f"Anomalias sin escalar: {int(pred_no_scale.sum())}")
print(f"Anomalias con escalar: {int(pred_scaled.sum())}")
print(f"Overlap: {overlap}")
print(f"Etiquetas distintas: {pct_diff}%")
print(f"\nRango original de cada KPI:")
print(X[SHORT_NAMES].describe().loc[["min", "max", "std"]])